# 2.1 · Data manipulation and tensors

**Machine Learning Foundations for AI Safety · Chapter 2**  
**Estimated time:** 30–40 minutes · **Prerequisite:** knowing what a Python list is.

### What you will be able to do

- Explain a tensor using everyday examples, identify its **shape** and **axes**, and tell a scalar, vector and matrix apart.
- Select a cell or a row/column using Python indexing.
- Explain the difference between *elementwise arithmetic* and *broadcasting*.
- Run small PyTorch examples and predict their outputs before running them.

> **Our running example:** A made-up class of four learners. Each learner has study hours, completed practice questions, and a quiz score. We use synthetic numbers for easy arithmetic; they are not evidence that hours alone determine a real student's score.

**Learning method:** Read the idea → inspect the picture → predict the output → run the code → explain it in your own words.


## 1. What does “data” look like to a computer?

We may write **“Ben studied 2 hours, completed 4 questions, and scored 5.”** A computer can process this observation as three numbers: `[2, 4, 5]`. Put four learners together and we get a rectangular **table**.

A **row** describes *one learner*. A **column** describes *one type of measurement*. A **cell** is where a particular row meets a particular column; every cell stores exactly one value.

Here is the entire table. Count the four learner rows and three numeric columns. The table therefore has shape **(4, 3)**. The names help humans read the data, but the names are not included in this particular numeric matrix.


<picture>
  <source type="image/svg+xml" srcset="images/01-table-rows-columns.svg">
  <img src="images/01-table-rows-columns.png" alt="A fully labeled 4-row × 3-column data table with the cell for Ben’s quiz score highlighted" loading="lazy">
</picture>

*Figure: A fully labeled 4-row × 3-column data table with the cell for Ben’s quiz score highlighted.*


For example, **row Ben + column Score = 5**. The table is not just a picture of cells: it is an organized way to tell the computer which value belongs where.

### Try this in Python

First build the numeric table with PyTorch, a library used widely in deep learning. We will not need a GPU or any downloaded dataset.


In [8]:
import torch

learner_names = ["Alice", "Ben", "Chika", "Dara"]
data = torch.tensor([
    [1., 2., 3.],  # Alice: hours, practice, score
    [2., 4., 5.],  # Ben
    [3., 6., 7.],  # Chika
    [4., 8., 9.],  # Dara
])

print('Shape:', tuple(data.shape))
print('Number of values:', data.numel())
print('First row:', data[0].tolist())
assert tuple(data.shape) == (4, 3)


Shape: (4, 3)
Number of values: 12
First row: [1.0, 2.0, 3.0]


**Read the result:** A shape `(4, 3)` means 4 rows and 3 columns; `numel()` returns **12** because 4 × 3 = 12. The individual data values have not changed—only the way we grouped them determines the shape.

## 2. What is a tensor?

A **tensor** is an array of values organized along one or more **axes** (directions). A scalar is a single number; a vector is a one-dimensional array; a matrix is a two-dimensional array. In PyTorch, all three are tensors.

- **Scalar:** one quiz score, `5`; shape `()` (zero axes).
- **Vector:** four scores, `[3, 5, 7, 9]`; shape `(4,)` (one axis).
- **Matrix:** all the numerical observations; shape `(4, 3)` (two axes).
- **Higher-dimensional tensor:** a group of color images can be organized as `(number_of_images, color_channels, height, width)`.

Think of an **axis** as a direction you can move through the data: down the rows (axis 0) or across the columns (axis 1).


### Why do we use tensors instead of separate numbers?

A model usually receives *many examples and many measurements at once*. A tensor stores those values in one organized object so an operation can be applied to an entire group. **Shape describes the arrangement, not the meaning of the values.**

For our dataset, `(4, 3)` means four learners × three numerical measurements. `shape[0] = 4` counts learners, while `shape[1] = 3` counts measurements per learner. A scalar has zero axes, a vector one, and a matrix two. Larger tensors follow the same idea: a batch of images might have shape `(batch, channels, height, width)`.

**Machine-learning connection:** if your model expects three features per example, providing a tensor with four columns will cause a shape mismatch—even if every number looks reasonable.


<picture>
  <source type="image/svg+xml" srcset="images/02-tensor-shapes.svg">
  <img src="images/02-tensor-shapes.png" alt="A scalar, a four-number vector, and a 2 × 3 matrix with every cell shown" loading="lazy">
</picture>

*Figure: A scalar, a four-number vector, and a 2 × 3 matrix with every cell shown.*


In [9]:
scalar = torch.tensor(5.)
vector = torch.tensor([3., 5., 7., 9.])
matrix = torch.tensor([[1., 2., 3.], [4., 5., 6.]])

for label, tensor in [('scalar', scalar), ('vector', vector), ('matrix', matrix)]:
    print(f'{label:6}: shape={tuple(tensor.shape)}, dimensions={tensor.ndim}')

# Reshaping changes the arrangement, not the individual values.
sequence = torch.arange(12)
print('Reshaped to 3 rows × 4 columns:\n', sequence.reshape(3, 4))


scalar: shape=(), dimensions=0
vector: shape=(4,), dimensions=1
matrix: shape=(2, 3), dimensions=2
Reshaped to 3 rows × 4 columns:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])


**Predict before running:** `torch.arange(12)` contains 12 numbers, starting at 0 and ending at 11. If we reshape it to `(3, 4)`, we get 3 rows of 4 numbers—not 12 new numbers. A reshape only works when the total number of entries is preserved.

## 3. Finding a particular value: indexing and slicing

Python starts counting positions at **0**, not 1. `data[1, 2]` means **second row, third column**. That is Ben’s score, `5`.

A colon (`:`) says “take all entries along this axis.” So `data[:, 0]` returns **the first column for every learner**. A slice like `data[0:2]` returns rows at positions 0 and 1, stopping *before* 2.


<picture>
  <source type="image/svg+xml" srcset="images/03-indexing.svg">
  <img src="images/03-indexing.png" alt="A rectangular tensor with explicit zero-based row and column indices and one highlighted entry" loading="lazy">
</picture>

*Figure: A rectangular tensor with explicit zero-based row and column indices and one highlighted entry.*


In [10]:
print('One cell:', data[1, 2].item())
print('All study hours:', data[:, 0].tolist())
print('Ben’s entire row:', data[1].tolist())
print('First two learners:\n', data[0:2])

assert data[1, 2].item() == 5
assert data[:, 0].tolist() == [1., 2., 3., 4.]


One cell: 5.0
All study hours: [1.0, 2.0, 3.0, 4.0]
Ben’s entire row: [2.0, 4.0, 5.0]
First two learners:
 tensor([[1., 2., 3.],
        [2., 4., 5.]])


## 4. Operations: elementwise vs. broadcasting

### First: what does *elementwise* mean?

With equally shaped tensors, `+` combines **corresponding positions**: `[1, 2] + [10, 20]` becomes `[11, 22]`. Similarly, `*` multiplies positions: `[1, 2] * [10, 20]` becomes `[10, 40]`. No rows or columns are matched by meaning or by name: PyTorch matches their **positions**.

### Broadcasting

**Broadcasting is a set of rules that lets PyTorch perform elementwise operations on tensors of different but compatible shapes by treating dimensions of size 1 as expanded to the required size.** 'Expand' means *behave as if values repeat*, not necessarily that PyTorch physically copied the tensor's data. Now follow the original calculation below, and then the expanded view after it.


<picture>
  <source type="image/svg+xml" srcset="images/04-broadcasting.svg">
  <img src="images/04-broadcasting.png" alt="A complete 3 × 1 plus 1 × 2 broadcasting calculation with a full 3 × 2 result table" loading="lazy">
</picture>

*Figure: A complete 3 × 1 plus 1 × 2 broadcasting calculation with a full 3 × 2 result table.*


### How the `(3, 1) + (1, 2)` addition works

**Step 1 — read the shapes.** `A = [[1], [2], [3]]` has **3 rows and 1 column**. `B = [[10, 20]]` has **1 row and 2 columns**.

**Step 2 — imagine expanding only the size-1 dimensions.** A's single column is logically repeated into two columns. B's single row is logically repeated into three rows. Their conceptual views are:

| Row | A column 0 | A column 1 | B column 0 | B column 1 |
|:--|--:|--:|--:|--:|
| 0 | 1 | 1 | 10 | 20 |
| 1 | 2 | 2 | 10 | 20 |
| 2 | 3 | 3 | 10 | 20 |

**Step 3 — add corresponding positions:** top-right `1+20=21`, middle-left `2+10=12`, bottom-right `3+20=23`. The complete output is `[[11,21],[12,22],[13,23]]`, shape `(3,2)`.

Here is the same expansion as fully labeled 3-row × 2-column grids:


<picture>
  <source type="image/svg+xml" srcset="images/04b-broadcasting-expanded.svg">
  <img src="images/04b-broadcasting-expanded.png" alt="Three complete three-row two-column grids: A expands horizontally, B expands vertically, and their elementwise sum is shown cell by cell." loading="lazy">
</picture>

*Figure: Conceptual broadcasting expansion. Colored cells show reused values; no physical copies are required.*


In [11]:
A = torch.tensor([[1.], [2.], [3.]])   # three rows, one column
B = torch.tensor([[10., 20.]])           # one row, two columns

result = A + B
print('A shape:', tuple(A.shape))
print('B shape:', tuple(B.shape))
print('Result shape:', tuple(result.shape))
print(result)

assert result.tolist() == [[11., 21.], [12., 22.], [13., 23.]]


A shape: (3, 1)
B shape: (1, 2)
Result shape: (3, 2)
tensor([[11., 21.],
        [12., 22.],
        [13., 23.]])


### Try a compatible shape, then see a shape error

In the cell below we use tiny stand-in values rather than 1,000 learners. Predict the output shape before running. A failed example is caught and reported so it does not stop the notebook.


In [12]:
subject_scores = torch.tensor([[60., 70., 80.], [75., 85., 95.]])  # 2 learners × 3 subjects
subject_means = torch.tensor([67.5, 77.5, 87.5])           # 3 subject averages
centered = subject_scores - subject_means
print('Scores shape:', tuple(subject_scores.shape))
print('Means shape:', tuple(subject_means.shape))
print('Centered result:\n', centered)
assert centered.shape == (2, 3)

try:
    _ = torch.zeros(3, 2) + torch.zeros(3, 3)
except RuntimeError:
    print('Expected: shapes (3,2) and (3,3) cannot broadcast.')


Scores shape: (2, 3)
Means shape: (3,)
Centered result:
 tensor([[-7.5000, -7.5000, -7.5000],
        [ 7.5000,  7.5000,  7.5000]])
Expected: shapes (3,2) and (3,3) cannot broadcast.


### When are shapes compatible? Check dimensions from right to left

PyTorch lines up the **last axes first**. Each aligned pair must have the **same size**, or **one size must be 1**, or a leading axis must be absent (treated as 1 for comparison). The result takes the larger size along each axis. If a pair has two different sizes, neither of which is 1, broadcasting fails.

| A shape | B shape | Does it work? | Result and reasoning |
|:--|:--|:--|:--|
| `(3, 1)` | `(1, 2)` | Yes | `(3, 2)` — each axis has a 1 |
| `(3, 2)` | `(3, 2)` | Yes | `(3, 2)` — both axes match |
| `(3, 2)` | `(2,)` | Yes | `(3, 2)` — the vector acts like `(1, 2)` |
| `(3, 2)` | `(3, 3)` | **No** | Last axis: 2 versus 3; neither is 1 |
| `(3, 1, 4)` | `(1, 5, 1)` | Yes | `(3, 5, 4)` — check axes from the right |

**Do not confuse operations:** `A * B` is elementwise multiplication, optionally with broadcasting. `A @ B` is *matrix multiplication*, which uses row–column dot products and its own shape rules (Lesson 2.3).

### Where broadcasting appears in machine learning

Suppose `scores` contains 1,000 learners × 3 subject scores, so it has shape `(1000, 3)`. Suppose `subject_means` contains the three averages, shape `(3,)`. `scores - subject_means` subtracts the relevant subject average from **each learner's row**, without manually creating 1,000 copies of the averages. This pattern is common in feature centering and neural-network bias addition.

**Check the intuition:** Broadcasting changes *how the computation pairs values*. It does **not** invent new observations or determine what the data means.


In [13]:
experiment = data.clone()  # we create a copy, leaving the original unchanged
experiment[1, 2] = 6.
print('Old Ben score:', data[1, 2].item())
print('Edited Ben score:', experiment[1, 2].item())
print('The edited copy differs from the original:', not torch.equal(data, experiment))  # True


Old Ben score: 5.0
Edited Ben score: 6.0
The edited copy differs from the original: True


### Explain broadcasting in your own words

Try this four-part explanation aloud: **What** does broadcasting allow? **Why** is it useful? **How** are dimensions compared? **Where** does it appear in machine learning? If you can explain all four using `(3,1) + (1,2)`, you understand the rule rather than memorizing the diagram.


### Check your understanding (answer before revealing by running)

1. A tensor has shape `(5, 4)`. How many **rows**, **columns**, and **values** are there?
2. Which expression selects Chika’s **hours** from `data`?
3. What is the result shape of `(2, 1)` + `(1, 3)`? What are the result’s dimensions?
4. Explain **in one sentence** why `*` is not the same as `@`.

**Practice cell:** Try `data[2, 0]`, `data[:, 1]`, and a broadcast of `[[5], [6]] + [[0, 10, 20]]`.

**Takeaway:** The value of a number and the *position* it occupies in a tensor are different ideas. Understanding shape and indexing prevents many later ML errors.

➡️ **Next:** [2.2 Data preprocessing](02-data-preprocessing.ipynb) — real tables often include missing values and categories.


In [14]:
# Your turn — edit these expressions and predict their outputs.
print('Chika’s hours:', data[2, 0].item())
print('All practice values:', data[:, 1].tolist())
print('Broadcast example:\n', torch.tensor([[5], [6]]) + torch.tensor([[0, 10, 20]]))


Chika’s hours: 3.0
All practice values: [2.0, 4.0, 6.0, 8.0]
Broadcast example:
 tensor([[ 5, 15, 25],
        [ 6, 16, 26]])
